# Laboratório pessoal: Motion Transfer e Object Swap

Protótipo com **Wan2.1-VACE-1.3B**, sem API paga. A inferência ainda não foi
validada em GPU neste projeto; podem ser necessários ajustes de memória ou compatibilidade.

No Colab gratuito: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**, se disponível.
A disponibilidade e o tempo gratuito dependem do Google. Se não houver GPU gratuita,
pare: este notebook não compra créditos nem cria máquinas pagas. Ele não elimina limites do Colab.

Seu Windows/RX 580 serve para acessar o navegador. A GPU que gera o vídeo é a da sessão Colab.
Os arquivos enviados ficam na sessão de nuvem; não são processados na RX 580.
Reserve espaço para vários GB de modelos e comece com 17 quadros (~1 segundo).

Execute as células na ordem. Não é necessário publicar o repositório ou fornecer chaves de API.


In [ ]:
import os, shutil
import torch
if not torch.cuda.is_available() or getattr(torch.version, "hip", None):
    raise RuntimeError("Esta sessão não tem GPU NVIDIA. Se o Colab não oferecer uma gratuitamente, pare aqui.")
if tuple(int(part) for part in torch.__version__.split('.')[:2]) < (2, 5):
    raise RuntimeError("Esta sessão usa PyTorch antigo. Use uma sessão Colab atual; não altere os drivers manualmente.")
if shutil.disk_usage('/content').free < 25 * 1024**3:
    raise RuntimeError("Libere pelo menos 25 GB no disco da sessão antes de baixar os modelos.")
print('GPU da sessão:', torch.cuda.get_device_name())
print('Memória da GPU (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print('A geração será testada nesta GPU, não na placa do seu computador.')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HOME'] = '/content/genjutsu-model-cache'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'


## Preparar o programa e instalar as dependências gratuitas


In [ ]:
from pathlib import Path
import json, os
root = Path('/content/genjutsu-pessoal')
root.mkdir(exist_ok=True)
embedded = json.loads('{"media.py": "\\"\\"\\"Video preparation only. This module never pretends to generate AI output.\\"\\"\\"\\nfrom pathlib import Path\\nimport math\\n\\nimport cv2\\nimport numpy as np\\nfrom PIL import Image, ImageDraw, ImageOps\\n\\nFPS = 16\\n\\n\\ndef normalize(image: Image.Image, size: tuple[int, int], color=\\"black\\") -> Image.Image:\\n    return ImageOps.pad(ImageOps.exif_transpose(image).convert(\\"RGB\\"), size,\\n                        method=Image.Resampling.LANCZOS, color=color)\\n\\n\\ndef read_clip(path, size=(384, 224), max_frames=17):\\n    if max_frames not in (17, 33, 49, 81):\\n        raise ValueError(\\"Escolha 17, 33, 49 ou 81 quadros.\\")\\n    if any(v < 64 or v % 16 for v in size):\\n        raise ValueError(\\"As dimensões precisam ser múltiplas de 16, com pelo menos 64 pixels.\\")\\n    path = Path(path)\\n    if not path.is_file() or path.stat().st_size > 100 * 1024**2:\\n        raise ValueError(\\"Envie um vídeo de até 100 MB.\\")\\n    cap = cv2.VideoCapture(str(path))\\n    try:\\n        rate = cap.get(cv2.CAP_PROP_FPS)\\n        count = cap.get(cv2.CAP_PROP_FRAME_COUNT)\\n        if not cap.isOpened() or not math.isfinite(rate) or rate <= 0 or count < 2:\\n            raise ValueError(\\"Não foi possível ler este vídeo. Experimente MP4/H.264.\\")\\n        # Last sample must be within the source. VACE requires 4*n + 1 frames.\\n        available = math.floor((count - 1) / rate * FPS + 1e-6) + 1\\n        n = min(max_frames, available)\\n        n = ((n - 1) // 4) * 4 + 1\\n        if n < 5:\\n            raise ValueError(\\"O vídeo precisa ter pelo menos cerca de meio segundo.\\")\\n        frames = []\\n        for i in range(n):\\n            source_index = min(int(round(i * rate / FPS)), int(count) - 1)\\n            cap.set(cv2.CAP_PROP_POS_FRAMES, source_index)\\n            ok, frame = cap.read()\\n            if not ok:\\n                raise ValueError(f\\"Não foi possível ler o quadro {source_index}.\\")\\n            frames.append(normalize(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), size))\\n        return frames\\n    finally:\\n        cap.release()\\n\\n\\ndef parse_box(value):\\n    try:\\n        box = tuple(float(x.strip()) for x in value.split(\\",\\"))\\n    except (TypeError, ValueError, AttributeError) as exc:\\n        raise ValueError(\\"Região: use quatro números, por exemplo 0.25,0.2,0.75,0.85.\\") from exc\\n    if (len(box) != 4 or not all(math.isfinite(x) and 0 <= x <= 1 for x in box)\\n            or box[0] >= box[2] or box[1] >= box[3]):\\n        raise ValueError(\\"Região inválida. Use esquerda,topo,direita,base entre 0 e 1.\\")\\n    return box\\n\\n\\ndef box_masks(size, count, start, end=None):\\n    start = parse_box(start)\\n    end = parse_box(end) if end else start\\n    width, height = size\\n    masks = []\\n    for i in range(count):\\n        fraction = i / max(count - 1, 1)\\n        box = [a + fraction * (b - a) for a, b in zip(start, end)]\\n        x0, y0 = int(box[0] * width), int(box[1] * height)\\n        x1, y1 = math.ceil(box[2] * width), math.ceil(box[3] * height)\\n        mask = Image.new(\\"L\\", size, 0)\\n        ImageDraw.Draw(mask).rectangle((x0, y0, x1 - 1, y1 - 1), fill=255)\\n        masks.append(mask)\\n    return masks\\n\\n\\ndef mask_preview(frame, mask):\\n    overlay = Image.new(\\"RGB\\", frame.size, \\"#ff5252\\")\\n    return Image.composite(Image.blend(frame, overlay, .45), frame, mask)\\n\\n\\ndef save_video(frames, path, fps=FPS):\\n    import imageio.v2 as imageio\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    with imageio.get_writer(str(path), fps=fps, codec=\\"libx264\\", macro_block_size=1,\\n                            ffmpeg_log_level=\\"error\\") as writer:\\n        for frame in frames:\\n            writer.append_data(np.asarray(frame.convert(\\"RGB\\"), dtype=np.uint8))\\n    cap = cv2.VideoCapture(str(path))\\n    try:\\n        if not cap.isOpened() or cap.get(cv2.CAP_PROP_FRAME_COUNT) != len(frames):\\n            raise RuntimeError(\\"O vídeo salvo não contém a sequência esperada.\\")\\n    finally:\\n        cap.release()\\n    return str(path)\\n\\n", "engine.py": "\\"\\"\\"Real VACE inference, gated on CUDA. No paid API or simulated generation.\\"\\"\\"\\nfrom datetime import datetime, timezone\\nimport gc\\nimport hashlib\\nfrom importlib.metadata import version, PackageNotFoundError\\nimport json\\nfrom pathlib import Path\\nimport time\\nimport uuid\\n\\nimport numpy as np\\nfrom PIL import Image\\n\\nfrom media import FPS, box_masks, normalize, read_clip, save_video\\n\\nMODEL = \\"Wan-AI/Wan2.1-VACE-1.3B-diffusers\\"\\nNEGATIVE = \\"blurry, low quality, distorted anatomy, extra limbs, text, watermark, flickering\\"\\n\\n\\ndef require_gpu():\\n    try:\\n        import torch\\n    except ImportError as exc:\\n        raise RuntimeError(\\"O motor não está instalado. Use o notebook em uma sessão com GPU NVIDIA.\\") from exc\\n    if not torch.cuda.is_available() or getattr(torch.version, \\"hip\\", None):\\n        raise RuntimeError(\\"Geração indisponível: este protótipo requer GPU NVIDIA/CUDA. \\"\\n                           \\"A RX 580/DirectML não foi validada. Nenhuma geração foi simulada.\\")\\n    if torch.cuda.get_device_capability()[0] < 7:\\n        raise RuntimeError(\\"A GPU é antiga para o caminho de quantização deste protótipo.\\")\\n    return torch\\n\\n\\ndef _hash(path):\\n    h = hashlib.sha256()\\n    with open(path, \\"rb\\") as file:\\n        for chunk in iter(lambda: file.read(1024 * 1024), b\\"\\"):\\n            h.update(chunk)\\n    return h.hexdigest()\\n\\n\\ndef encode_text(torch, prompt, dtype):\\n    \\"\\"\\"Encode first, release the text model, then load the video generator.\\n\\n    4-bit text encoding reduces memory; this is an experimental configuration,\\n    not a claim about a minimum GPU size or equivalence to the full model.\\n    \\"\\"\\"\\n    from diffusers import WanVACEPipeline, UniPCMultistepScheduler\\n    from transformers import AutoTokenizer, UMT5EncoderModel, BitsAndBytesConfig\\n    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\\"nf4\\",\\n                              bnb_4bit_compute_dtype=dtype)\\n    text_encoder = UMT5EncoderModel.from_pretrained(\\n        MODEL, subfolder=\\"text_encoder\\", quantization_config=quant,\\n        device_map={\\"\\": 0}, torch_dtype=dtype)\\n    tokenizer = AutoTokenizer.from_pretrained(MODEL, subfolder=\\"tokenizer\\")\\n    pipe = WanVACEPipeline(tokenizer=tokenizer, text_encoder=text_encoder, vae=None,\\n                           transformer=None, scheduler=UniPCMultistepScheduler(flow_shift=3.0))\\n    try:\\n        with torch.inference_mode():\\n            positive, negative = pipe.encode_prompt(\\n                prompt=prompt, negative_prompt=NEGATIVE, device=torch.device(\\"cuda\\"),\\n                do_classifier_free_guidance=True, max_sequence_length=256)\\n        return positive.cpu(), negative.cpu()\\n    finally:\\n        del pipe, text_encoder, tokenizer\\n        gc.collect()\\n        torch.cuda.empty_cache()\\n\\n\\ndef prepare_pose(frames, report):\\n    from controlnet_aux import OpenposeDetector\\n    detector = OpenposeDetector.from_pretrained(\\"lllyasviel/Annotators\\")\\n    try:\\n        result = []\\n        for i, frame in enumerate(frames):\\n            report(f\\"Extraindo pose: {i + 1}/{len(frames)}\\")\\n            pose = detector(frame, include_body=True, include_hand=False, include_face=False)\\n            if pose.convert(\\"L\\").getbbox() is None:\\n                raise ValueError(f\\"Nenhuma pose detectada no quadro {i + 1}. Use um vídeo com uma pessoa visível.\\")\\n            result.append(pose.resize(frame.size, Image.Resampling.BILINEAR).convert(\\"RGB\\"))\\n        return result\\n    finally:\\n        del detector\\n        gc.collect()\\n\\n\\ndef generate(video, reference, mode, prompt, size=(384, 224), max_frames=17,\\n             steps=25, seed=42, start_box=\\"0.25,0.2,0.75,0.85\\", end_box=\\"\\",\\n             output_root=\\"outputs\\", report=lambda message: None):\\n    if mode not in (\\"motion\\", \\"swap\\"):\\n        raise ValueError(\\"Modo desconhecido.\\")\\n    if not prompt or not prompt.strip():\\n        raise ValueError(\\"Descreva a aparência e a ação desejadas.\\")\\n    if not 10 <= int(steps) <= 50 or not 0 <= int(seed) < 2**32:\\n        raise ValueError(\\"Passos ou semente fora do intervalo permitido.\\")\\n    # Validate inputs before downloading any model or touching a GPU.\\n    frames = read_clip(video, size=size, max_frames=int(max_frames))\\n    with Image.open(reference) as im:\\n        ref = normalize(im, size, color=\\"white\\")\\n    masks = box_masks(size, len(frames), start_box, end_box) if mode == \\"swap\\" else [\\n        Image.new(\\"L\\", size, 0) for _ in frames]\\n    torch = require_gpu()\\n    run = Path(output_root).resolve() / uuid.uuid4().hex\\n    run.mkdir(parents=True)\\n    metadata = {\\"status\\": \\"running\\", \\"created_at\\": datetime.now(timezone.utc).isoformat(),\\n                \\"model\\": MODEL, \\"mode\\": mode, \\"prompt\\": prompt, \\"negative_prompt\\": NEGATIVE,\\n                \\"seed\\": int(seed), \\"steps\\": int(steps), \\"fps\\": FPS,\\n                \\"size\\": list(size), \\"frames\\": len(frames), \\"start_box\\": start_box if mode == \\"swap\\" else None,\\n                \\"end_box\\": end_box if mode == \\"swap\\" else None,\\n                \\"input_sha256\\": _hash(video), \\"reference_sha256\\": _hash(reference),\\n                \\"gpu\\": torch.cuda.get_device_name(), \\"text_quantization\\": \\"nf4\\",\\n                \\"audio\\": \\"omitted\\", \\"paid_inference_api_calls\\": 0}\\n    for package in (\\"torch\\", \\"diffusers\\", \\"transformers\\", \\"bitsandbytes\\", \\"controlnet-aux\\"):\\n        try:\\n            metadata[package] = version(package)\\n        except PackageNotFoundError:\\n            pass\\n    manifest = run / \\"run.json\\"\\n    started = time.monotonic()\\n    pipe = None\\n    try:\\n        manifest.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))\\n        report(\\"Preparando sinais de controle\\")\\n        controls = prepare_pose(frames, report) if mode == \\"motion\\" else frames\\n        save_video(controls, run / \\"controle.mp4\\")\\n        save_video([m.convert(\\"RGB\\") for m in masks], run / \\"mascara.mp4\\")\\n        ref.save(run / \\"referencia.png\\")\\n        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\\n        report(\\"Carregando o codificador de texto; o primeiro uso baixa os modelos\\")\\n        positive, negative = encode_text(torch, prompt, dtype)\\n        from diffusers import AutoencoderKLWan, WanVACEPipeline, UniPCMultistepScheduler\\n        report(\\"Carregando o gerador VACE\\")\\n        vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder=\\"vae\\", torch_dtype=torch.float32)\\n        pipe = WanVACEPipeline.from_pretrained(MODEL, vae=vae, text_encoder=None,\\n                                               tokenizer=None, torch_dtype=dtype)\\n        pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)\\n        pipe.vae.enable_tiling()\\n        pipe.enable_model_cpu_offload()\\n\\n        def on_step(pipeline, step, timestep, kwargs):\\n            report(f\\"Gerando: etapa {step + 1}/{steps}\\")\\n            return kwargs\\n\\n        with torch.inference_mode():\\n            output = pipe(video=controls, mask=masks, reference_images=[ref],\\n                          prompt_embeds=positive, negative_prompt_embeds=negative,\\n                          height=size[1], width=size[0], num_frames=len(frames),\\n                          num_inference_steps=int(steps), guidance_scale=5.0,\\n                          generator=torch.Generator(device=\\"cpu\\").manual_seed(int(seed)),\\n                          output_type=\\"np\\", callback_on_step_end=on_step).frames[0]\\n        if len(output) != len(frames):\\n            raise RuntimeError(\\"O modelo retornou uma quantidade inesperada de quadros.\\")\\n        if not np.isfinite(output).all():\\n            raise RuntimeError(\\"A geração apresentou erro numérico. Nenhum vídeo inválido será entregue.\\")\\n        output = [Image.fromarray((np.clip(frame, 0, 1) * 255).round().astype(np.uint8)) for frame in output]\\n        report(\\"Salvando o vídeo gerado\\")\\n        result = save_video(output, run / \\"resultado.mp4\\")\\n        metadata[\\"status\\"] = \\"completed\\"\\n        return result, str(manifest)\\n    except Exception as exc:\\n        metadata.update(status=\\"failed\\", error=f\\"{type(exc).__name__}: {exc}\\")\\n        raise\\n    finally:\\n        metadata[\\"elapsed_seconds\\"] = round(time.monotonic() - started, 2)\\n        manifest.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))\\n        if pipe is not None:\\n            del pipe\\n        gc.collect()\\n        torch.cuda.empty_cache()\\n\\n", "app.py": "import os\\n\\nos.environ.setdefault(\\"GRADIO_ANALYTICS_ENABLED\\", \\"False\\")\\n\\nimport gradio as gr\\nfrom engine import generate\\nfrom media import box_masks, mask_preview, read_clip\\n\\nSIZES = {\\"Rascunho — 256 × 144\\": (256, 144), \\"384 × 224\\": (384, 224),\\n         \\"480 × 272\\": (480, 272)}\\n\\n\\ndef preview(video, resolution, count, start, end):\\n    if not video:\\n        raise gr.Error(\\"Envie um vídeo primeiro.\\")\\n    try:\\n        size = SIZES[resolution]\\n        frames = read_clip(video, size, int(count))\\n        masks = box_masks(size, len(frames), start, end)\\n        return mask_preview(frames[0], masks[0]), mask_preview(frames[-1], masks[-1])\\n    except (ValueError, OSError) as exc:\\n        raise gr.Error(str(exc)) from exc\\n\\n\\ndef run(video, reference, mode, prompt, resolution, count, steps, seed, start, end,\\n        progress=gr.Progress()):\\n    if not video or not reference:\\n        raise gr.Error(\\"Envie o vídeo e a imagem de referência.\\")\\n    try:\\n        return generate(video, reference, mode, prompt, SIZES[resolution], int(count),\\n                        int(steps), int(seed), start, end,\\n                        report=lambda message: progress(None, desc=message))\\n    except Exception as exc:\\n        raise gr.Error(str(exc)) from exc\\n\\n\\ndef build_app():\\n    with gr.Blocks(title=\\"Genjutsu — laboratório pessoal\\") as demo:\\n        gr.Markdown(\\"# Laboratório pessoal de vídeo\\\\n\\"\\n                    \\"Modelo aberto VACE, sem API paga. **Protótipo experimental: a geração ainda \\"\\n                    \\"precisa ser validada em GPU.** Requer NVIDIA/CUDA; não há suporte validado para RX 580. \\"\\n                    \\"Comece com 17 quadros. A primeira execução baixa vários GB de modelos.\\")\\n        with gr.Row():\\n            video = gr.Video(label=\\"Vídeo de movimento ou cena\\", sources=[\\"upload\\"])\\n            reference = gr.Image(label=\\"Imagem do personagem ou objeto substituto\\", type=\\"filepath\\", sources=[\\"upload\\"])\\n        mode = gr.Radio([(\\"Transferir movimento corporal\\", \\"motion\\"),\\n                         (\\"Trocar objeto na região marcada\\", \\"swap\\")], value=\\"motion\\", label=\\"Função\\")\\n        prompt = gr.Textbox(label=\\"Descreva o resultado desejado (preferencialmente em inglês)\\",\\n                            placeholder=\\"A person wearing a red jacket raises their arms.\\")\\n        with gr.Row():\\n            resolution = gr.Dropdown(list(SIZES), value=\\"Rascunho — 256 × 144\\", label=\\"Resolução\\")\\n            count = gr.Dropdown([17, 33, 49, 81], value=17, label=\\"Quadros: aproximadamente 1, 2, 3 ou 5 segundos\\")\\n        with gr.Accordion(\\"Região para troca de objeto\\", open=False):\\n            gr.Markdown(\\"Use esquerda, topo, direita, base entre 0 e 1. Por exemplo, \\"\\n                        \\"0.25,0.2,0.75,0.85. A área vermelha será editada. \\"\\n                        \\"Os números se referem à prévia redimensionada. Não há rastreamento automático: \\"\\n                        \\"se indicar duas regiões, o retângulo se move em linha reta entre elas.\\")\\n            start = gr.Textbox(value=\\"0.25,0.2,0.75,0.85\\", label=\\"Região no início\\")\\n            end = gr.Textbox(value=\\"\\", label=\\"Região no fim (vazio = região fixa)\\")\\n            preview_button = gr.Button(\\"Conferir região antes de gerar\\")\\n            with gr.Row():\\n                first = gr.Image(label=\\"Primeiro quadro\\", interactive=False)\\n                last = gr.Image(label=\\"Último quadro do trecho\\", interactive=False)\\n            preview_button.click(preview, [video, resolution, count, start, end], [first, last])\\n        with gr.Accordion(\\"Ajustes\\", open=False):\\n            steps = gr.Slider(10, 50, value=25, step=1, label=\\"Etapas de geração\\")\\n            seed = gr.Number(value=42, precision=0, minimum=0, maximum=2**32-1, label=\\"Semente\\")\\n        gr.Markdown(\\"Limites: uma pessoa no modo movimento; só movimento corporal, sem transferência facial. \\"\\n                    \\"A troca de objeto depende da região fornecida. Resultados podem deformar ou alterar \\"\\n                    \\"o fundo. Saída sem áudio. Nenhuma promessa de qualidade equivalente ao Genjutsu.\\")\\n        submit = gr.Button(\\"Gerar vídeo\\", variant=\\"primary\\")\\n        result = gr.Video(label=\\"Resultado gerado\\", interactive=False)\\n        manifest = gr.File(label=\\"Registro da execução\\")\\n        submit.click(run, [video, reference, mode, prompt, resolution, count, steps, seed, start, end],\\n                     [result, manifest], concurrency_limit=1, concurrency_id=\\"generation\\")\\n    return demo.queue(max_size=2)\\n\\n\\nif __name__ == \\"__main__\\":\\n    build_app().launch(server_name=\\"127.0.0.1\\", share=False, max_file_size=\\"100mb\\")\\n", "requirements.txt": "gradio==5.49.1\\nhuggingface-hub==0.36.0\\nnumpy==1.26.4\\nPillow==11.3.0\\nopencv-python-headless==4.11.0.86\\nimageio==2.37.0\\nimageio-ffmpeg==0.6.0\\n\\n", "requirements-model.txt": "-r requirements.txt\\ntorch>=2.5,<3\\ntorchvision>=0.20\\ndiffusers==0.35.2\\ntransformers==4.57.6\\naccelerate==1.12.0\\nbitsandbytes==0.48.2\\ncontrolnet-aux==0.0.10\\nsentencepiece==0.2.1\\nftfy==6.3.1\\n\\n"}')
for name, source in embedded.items():
    destination = root / name
    if destination.exists() and destination.read_text() != source:
        raise RuntimeError(f'{destination} contém alterações. Preserve-as antes de substituir.')
    destination.write_text(source)
os.chdir(root)
print('Programa preparado em', root)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-model.txt'], check=True)
from importlib.metadata import version
changed = []
for module_name, package_name in [('numpy', 'numpy'), ('PIL', 'Pillow'), ('torch', 'torch'),
                                  ('huggingface_hub', 'huggingface-hub')]:
    module = sys.modules.get(module_name)
    if module is not None and getattr(module, '__version__', None) != version(package_name):
        changed.append(package_name)
if changed:
    raise RuntimeError('Dependências atualizadas: ' + ', '.join(changed) +
                       '. Reinicie a sessão (sem excluir os arquivos) e execute as células novamente.')
# O notebook não inicia servidores nem túneis públicos.
print('Dependências instaladas. A geração ainda precisa ser executada.')


## Enviar um vídeo e uma imagem
Envie arquivos próprios. O resultado inicial não terá áudio.


In [ ]:
from google.colab import files
from pathlib import Path
import uuid
upload_dir = root / 'uploads' / uuid.uuid4().hex
upload_dir.mkdir(parents=True)

def upload_one(label, allowed):
    print(label)
    received = files.upload()
    if len(received) != 1:
        raise ValueError('Envie exatamente um arquivo por vez.')
    name, data = next(iter(received.items()))
    safe_name = Path(name).name
    if Path(safe_name).suffix.lower() not in allowed or len(data) > 100 * 1024**2:
        raise ValueError('Formato não aceito ou arquivo maior que 100 MB.')
    destination = upload_dir / safe_name
    if destination.exists():
        destination = upload_dir / (uuid.uuid4().hex + Path(safe_name).suffix)
    destination.write_bytes(data)
    return str(destination)

video_path = upload_one('1. Escolha o vídeo:', {'.mp4', '.mov', '.webm', '.avi'})
reference_path = upload_one('2. Escolha a imagem do personagem/objeto:', {'.png', '.jpg', '.jpeg', '.webp'})


## Escolher a transformação

- **motion**: seguir os movimentos corporais; use uma pessoa bem visível. Não transfere expressão facial.
- **swap**: regenerar a região indicada usando a imagem do substituto. Comece com objeto parado.

Região = esquerda, topo, direita, base, entre 0 e 1, referida à prévia redimensionada.
Deixe a região final vazia para manter uma caixa fixa. Duas caixas produzem
deslocamento linear, não rastreamento inteligente ou tratamento especial de oclusões.


In [ ]:
mode = 'motion' #@param ['motion', 'swap']
prompt = 'A person wearing a red jacket raises their arms.' #@param {type:'string'}
frames = 17 #@param [17, 33, 49, 81] {type:'raw'}
steps = 25 #@param {type:'slider', min:10, max:50, step:1}
seed = 42 #@param {type:'integer'}
start_box = '0.25,0.2,0.75,0.85' #@param {type:'string'}
end_box = '' #@param {type:'string'}
size = (256, 144) # Rascunho de baixo consumo; a qualidade será limitada.


In [ ]:
import sys
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from media import read_clip, box_masks, mask_preview
from IPython.display import display
clip = read_clip(video_path, size, frames)
if mode == 'swap':
    masks = box_masks(size, len(clip), start_box, end_box)
    print('Primeiro e último quadro: a área vermelha será editada. Ajuste a região antes de gerar.')
    display(mask_preview(clip[0], masks[0]), mask_preview(clip[-1], masks[-1]))
else:
    print('Primeiro quadro do trecho de movimento:')
    display(clip[0])


## Gerar e baixar

O primeiro uso baixa o VACE e, no modo movimento, o detector de pose.
Este passo executa o modelo real. Uma falha não será substituída por vídeo de exemplo.
Se faltar memória, reduza para 17 quadros. Não há tempo de processamento garantido.
Baixe seus resultados antes de encerrar a sessão; os arquivos do Colab são temporários.


In [ ]:
from engine import generate
from IPython.display import Video, display
result, manifest = generate(video_path, reference_path, mode, prompt, size, frames,
                            steps, seed, start_box, end_box, report=print)
display(Video(result, embed=True))
print('Registro:', manifest)
files.download(result)
files.download(manifest)
